# D7: Bijection Reconciliation — HCD ↔ CPRA

Row-level reconciliation between Berkeley's HCD APR Table A2 submission and
D5's CPRA-derived output, for every CY year 2018–2025. Codifies the two-tier
bijection first constructed inline on 2026-05-28.

- Bug-fix context: `docs/audit/2026-05-28_adu_diagnostic.md`
- Methodology + CY2024 ledger: `data/audit/cy2024_reconciliation/README.md`

**Tiers:** (1) tracking-ID equality with normalization; (2) APN equality
(current then prior). Injective: no HCD row maps to two CPRA permits, no CPRA
permit reused. Remaining HCD rows are classified
(multi_row_same_apn / year_shifted_to_{yr} / no_cpra_presence).

Outputs per year: `data/audit/cy{year}_reconciliation/` with
matched_pairs.csv, h_unmatched.csv, c_unmatched.csv, README.md
(+ dedup_audit.csv for CY2025).

In [1]:
# Cell 2 — setup: imports, paths, validation, snapshot of committed CY2024 ledger
import pandas as pd
import sqlite3
import re
from pathlib import Path
from datetime import datetime
from collections import defaultdict

ROOT = Path("/Users/johngage/berkeley-data")
HCD_DB = ROOT / "databases" / "hcd_apr_mirror.db"
D5_DIR = ROOT / "output" / "D5"
AUDIT_BASE = ROOT / "data" / "audit"
CPRA_DIR = ROOT / "data" / "raw" / "cpra-downloads"
CPRA_2018 = CPRA_DIR / "BP_Annual Permit Report-2018-2022.xlsx"
CPRA_2023 = CPRA_DIR / "BP_Annual Permit Report-2023-2025.xlsx"
D5_AUDIT = D5_DIR / "audit_CY2018-2025.csv"
D7_OUT = ROOT / "output" / "D7"
D7_OUT.mkdir(parents=True, exist_ok=True)

# input validation (D6 pattern)
if not HCD_DB.exists():
    raise FileNotFoundError(f"{HCD_DB} not found. Build: python scripts/build_hcd_mirror.py")
for y in range(2018, 2026):
    p = D5_DIR / f"table_a2_CY{y}.csv"
    if not p.exists():
        raise FileNotFoundError(f"Missing D5 CSV: {p}. Run D5_apr_from_cpra.ipynb first.")
for p in (CPRA_2018, CPRA_2023, D5_AUDIT):
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")

# Snapshot the committed CY2024 ledger BEFORE the loop overwrites it (for the regression test).
# Old files use the iteration-internal _t2 suffix; new files drop it.
OLD_2024 = {}
old_dir = AUDIT_BASE / "cy2024_reconciliation"
for newname, oldname in [("matched_pairs", "matched_pairs"),
                         ("h_unmatched", "h_unmatched_t2"),
                         ("c_unmatched", "c_unmatched_t2")]:
    fp = old_dir / f"{oldname}.csv"
    OLD_2024[newname] = pd.read_csv(fp) if fp.exists() else None
print("Snapshotted committed CY2024 ledger:",
      {k: (None if v is None else len(v)) for k, v in OLD_2024.items()})
print(f"Paths OK. AUDIT_BASE: {AUDIT_BASE}")

Snapshotted committed CY2024 ledger: {'matched_pairs': 182, 'h_unmatched': 8, 'c_unmatched': 974}
Paths OK. AUDIT_BASE: /Users/johngage/berkeley-data/data/audit


## Load and normalize HCD mirror

In [2]:
# Cell 4 — load HCD mirror (Berkeley), D6 coercions. rowid AS hcd_row_id preserves
# the stable row identity used by the committed CY2024 ledger.
con = sqlite3.connect(f"file:{HCD_DB}?mode=ro&immutable=1", uri=True)
hcd_all = pd.read_sql_query(
    "SELECT rowid AS hcd_row_id, * FROM table_a2 WHERE JURIS_NAME='BERKELEY'", con)
con.close()

hcd_all["YEAR"] = pd.to_numeric(hcd_all["YEAR"], errors="coerce").astype("Int64")
for c in ["ENT_APPROVE_DT1", "BP_ISSUE_DT1", "CO_ISSUE_DT1"]:
    if c in hcd_all.columns:
        hcd_all[c] = pd.to_datetime(hcd_all[c], errors="coerce")

print(f"Total Berkeley rows (pre-dedup): {len(hcd_all)}")
print(f"Years present: {sorted(hcd_all['YEAR'].dropna().unique().tolist())}")

Total Berkeley rows (pre-dedup): 2170
Years present: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


## CY 2025 dedup

In [3]:
# Cell 6 — CY 2025 dedup (verbatim D6 logic): groupby 5-key, keep lowest index.
hcd_2025 = hcd_all[hcd_all["YEAR"] == 2025].copy()
print(f"CY 2025 pre-dedup: {len(hcd_2025)} rows")

dup_groups = hcd_2025.groupby(
    ["APN", "STREET_ADDRESS", "JURS_TRACKING_ID", "BP_ISSUE_DT1", "CO_ISSUE_DT1"],
    dropna=False,
)
clusters_audit = []
keep_idx = set()
for key, grp in dup_groups:
    sorted_idx = sorted(grp.index)
    kept = sorted_idx[0]
    keep_idx.add(kept)
    if len(grp) > 1:
        for ix in sorted_idx:
            clusters_audit.append({
                "row_index": ix, "kept_or_dropped": "kept" if ix == kept else "dropped",
                "cluster_size": len(grp),
                "APN": key[0], "STREET_ADDRESS": key[1], "JURS_TRACKING_ID": key[2],
                "BP_ISSUE_DT1": key[3], "CO_ISSUE_DT1": key[4],
            })

cy2025_dir = AUDIT_BASE / "cy2025_reconciliation"
cy2025_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame(clusters_audit).to_csv(cy2025_dir / "dedup_audit.csv", index=False)

hcd = pd.concat([hcd_all[hcd_all["YEAR"] != 2025], hcd_2025.loc[sorted(keep_idx)]],
                ignore_index=True)
print(f"CY 2025 post-dedup: {(hcd['YEAR'] == 2025).sum()} rows")
print(f"HCD total post-dedup: {len(hcd)} rows")
print(f"dedup_audit.csv: {len(clusters_audit)} dup-cluster member rows")

CY 2025 pre-dedup: 474 rows
CY 2025 post-dedup: 234 rows
HCD total post-dedup: 1930 rows
dedup_audit.csv: 474 dup-cluster member rows


## Bijection construction — year loop

In [4]:
# Cell 8 — two-tier injective bijection per year, with classification of the remainder.

# --- normalization helpers (match the 2026-05-28 construction exactly) ---
def norm_track(s):
    if not isinstance(s, str): return None
    s = re.sub(r"\s+", "", s.strip().upper())
    s = re.sub(r"-(REV|DEF)\d*$", "", s)
    return s or None

def norm_apn(s):
    if not isinstance(s, str): return None
    s = re.sub(r"[^0-9a-zA-Z]", "", s).lower()
    return s or None

CO7 = ["CO_VLOW_INCOME_DR","CO_VLOW_INCOME_NDR","CO_LOW_INCOME_DR","CO_LOW_INCOME_NDR",
       "CO_MOD_INCOME_DR","CO_MOD_INCOME_NDR","CO_ABOVE_MOD_INCOME"]
BP7 = ["BP_VLOW_INCOME_DR","BP_VLOW_INCOME_NDR","BP_LOW_INCOME_DR","BP_LOW_INCOME_NDR",
       "BP_MOD_INCOME_DR","BP_MOD_INCOME_NDR","BP_ABOVE_MOD_INCOME"]
MATCHED_COLS = ["hcd_row_id","cpra_permit_number","address_hcd","address_cpra","tracking_id",
                "co_units_hcd","co_units_d5","bp_units_hcd","bp_units_d5",
                "agree_co","agree_bp","units_agree","match_tier","in_c1","cpra_year"]

# --- full CPRA corpus APN set (all years) + D5 audit for routed-year, loaded once ---
def _load_cpra_apns():
    fa = pd.read_excel(CPRA_2018, sheet_name="BP_Annual Permit Report", header=7)
    fb = pd.read_excel(CPRA_2023, sheet_name="BP_Annual Permit Report", header=7)
    cf = pd.concat([fa, fb], ignore_index=True)
    return set(cf["Parcel Number"].apply(norm_apn).dropna())
cf_apns = _load_cpra_apns()
print(f"CPRA full-corpus distinct APNs: {len(cf_apns)}")

_audit = pd.read_csv(D5_AUDIT)
_audit["apn_n"] = _audit["apn"].apply(norm_apn)
audit_by_apn = {}
for _, r in _audit.iterrows():
    if pd.notna(r["apn_n"]) and r["apn_n"] not in audit_by_apn:
        ry = r["reporting_year_co"] if pd.notna(r["reporting_year_co"]) else r["reporting_year_bp"]
        audit_by_apn[r["apn_n"]] = int(ry) if pd.notna(ry) else None

def make_pair(h, c, year, tier, tracking_id):
    co_h, bp_h = float(h["co_units_hcd"]), float(h["bp_units_hcd"])
    co_d, bp_d = float(c["co_d5"]), float(c["bp_d5"])
    aco, abp = abs(co_h - co_d) < 0.5, abs(bp_h - bp_d) < 0.5
    return {
        "hcd_row_id": h["hcd_row_id"], "cpra_permit_number": c["JURS_TRACKING_ID"],
        "address_hcd": h["STREET_ADDRESS"], "address_cpra": c["STREET_ADDRESS"],
        "tracking_id": tracking_id,
        "co_units_hcd": co_h, "co_units_d5": co_d, "bp_units_hcd": bp_h, "bp_units_d5": bp_d,
        "agree_co": aco, "agree_bp": abp, "units_agree": (aco and abp),
        "match_tier": tier, "in_c1": True, "cpra_year": year,
    }

summary_rows = []
for year in range(2018, 2026):
    h_year = hcd[hcd["YEAR"] == year].copy()
    c_year = pd.read_csv(D5_DIR / f"table_a2_CY{year}.csv")

    for col in CO7 + BP7:
        h_year[col] = pd.to_numeric(h_year[col], errors="coerce").fillna(0)
    h_year["co_units_hcd"] = h_year[CO7].sum(axis=1)
    h_year["bp_units_hcd"] = h_year[BP7].sum(axis=1)
    h_year["tnorm"]    = h_year["JURS_TRACKING_ID"].apply(norm_track)
    h_year["apn_cur"]  = h_year["APN"].apply(norm_apn)
    h_year["apn_prior"]= h_year["PRIOR_APN"].apply(norm_apn) if "PRIOR_APN" in h_year.columns else None

    c_year["co_d5"] = pd.to_numeric(c_year["CO_ABOVE_MOD_INCOME"], errors="coerce").fillna(0)
    c_year["bp_d5"] = pd.to_numeric(c_year["BP_ABOVE_MOD_INCOME"], errors="coerce").fillna(0)
    c_year["tnorm"] = c_year["JURS_TRACKING_ID"].apply(norm_track)
    c_year["apn_n"] = c_year["APN"].apply(norm_apn)
    c_all_apns = set(c_year["apn_n"].dropna())

    # --- Tier 1: tracking-ID equality, injective ---
    c_by_t = defaultdict(list)
    for i, r in c_year.iterrows():
        if r["tnorm"]: c_by_t[r["tnorm"]].append(i)
    used, pairs, unmatched_h = set(), [], []
    for _, h in h_year.iterrows():
        t, assigned = h["tnorm"], None
        if t and t in c_by_t:
            for ci in c_by_t[t]:
                if ci not in used: assigned = ci; break
        if assigned is not None:
            used.add(assigned)
            pairs.append(make_pair(h, c_year.loc[assigned], year, "tier_1_tracking_id", t))
        else:
            unmatched_h.append(h)

    # --- Tier 2: APN current then prior, against still-unused C permits ---
    c_by_apn = defaultdict(list)
    for i, r in c_year.iterrows():
        if r["apn_n"]: c_by_apn[r["apn_n"]].append(i)
    rest = []
    for h in unmatched_h:
        found, tier = None, None
        for apn, tlabel in [(h["apn_cur"], "tier_2_apn_current"),
                            (h["apn_prior"], "tier_2_apn_prior")]:
            if apn and apn in c_by_apn:
                for ci in c_by_apn[apn]:
                    if ci not in used: found, tier = ci, tlabel; break
            if found is not None: break
        if found is not None:
            used.add(found)
            c = c_year.loc[found]
            pairs.append(make_pair(h, c, year, tier,
                                   f"{h['JURS_TRACKING_ID']} / {c['JURS_TRACKING_ID']}"))
        else:
            rest.append(h)

    # --- classify the remainder ---
    no_presence, n_multirow, yearshift = [], 0, {}
    for h in rest:
        in_c = (h["apn_cur"] in c_all_apns) or (h.get("apn_prior") in c_all_apns)
        if in_c:
            n_multirow += 1
        else:
            apn2 = h["apn_cur"] if h["apn_cur"] in cf_apns else (
                   h.get("apn_prior") if h.get("apn_prior") in cf_apns else None)
            if apn2:
                ry = audit_by_apn.get(apn2)
                yearshift[ry] = yearshift.get(ry, 0) + 1
            else:
                no_presence.append(h)

    # --- write outputs ---
    outdir = AUDIT_BASE / f"cy{year}_reconciliation"
    outdir.mkdir(parents=True, exist_ok=True)
    matched_df = pd.DataFrame(pairs, columns=MATCHED_COLS)
    matched_df.to_csv(outdir / "matched_pairs.csv", index=False)

    h_un = pd.DataFrame([{
        "hcd_row_id": h["hcd_row_id"], "JURS_TRACKING_ID": h["JURS_TRACKING_ID"],
        "STREET_ADDRESS": h["STREET_ADDRESS"], "APN": h["APN"],
        "PRIOR_APN": h.get("PRIOR_APN"), "UNIT_CAT": h.get("UNIT_CAT"),
        "co_units_hcd": h["co_units_hcd"], "bp_units_hcd": h["bp_units_hcd"],
        "cpra_presence": "no_cpra_presence",
    } for h in no_presence], columns=["hcd_row_id","JURS_TRACKING_ID","STREET_ADDRESS","APN",
        "PRIOR_APN","UNIT_CAT","co_units_hcd","bp_units_hcd","cpra_presence"])
    h_un.to_csv(outdir / "h_unmatched.csv", index=False)

    c_un = (c_year[~c_year.index.isin(used)][["JURS_TRACKING_ID","STREET_ADDRESS","APN",
            "UNIT_CAT","co_d5","bp_d5"]]
            .rename(columns={"co_d5":"co_units_d5","bp_d5":"bp_units_d5"}))
    c_un.to_csv(outdir / "c_unmatched.csv", index=False)

    # --- per-year README ---
    matched_co = matched_df["co_units_hcd"].sum() if len(matched_df) else 0
    matched_bp = matched_df["bp_units_hcd"].sum() if len(matched_df) else 0
    h_co_total = h_year["co_units_hcd"].sum()
    h_bp_total = h_year["bp_units_hcd"].sum()
    ys_str = ", ".join(f"{k}:{v}" for k, v in sorted(yearshift.items(), key=lambda x: (x[0] is None, x[0])))
    (outdir / "README.md").write_text(f"""# CY {year} Reconciliation Ledger

Auto-generated by D7_bijection.ipynb on {datetime.now():%Y-%m-%d}.
Row-level bijection: Berkeley HCD Table A2 ({len(h_year)} rows) ↔ D5 CY{year}
output ({len(c_year)} master permits).

## Files
- `matched_pairs.csv` ({len(matched_df)} rows): HCD↔CPRA matches with match_tier
  (tier_1_tracking_id / tier_2_apn_current / tier_2_apn_prior), side-by-side units,
  and units_agree flags.
- `h_unmatched.csv` ({len(h_un)} rows): HCD rows with no CPRA presence in any year
  (entitlement-stage ZP/PLN tracking IDs).
- `c_unmatched.csv` ({len(c_un)} rows): D5 CY{year} permits with no HCD match.
{"- `dedup_audit.csv`: CY2025 draft+final dedup record." if year == 2025 else ""}

## HCD unit accounting (7 affordability columns per stage)
- Matched HCD units: CO {matched_co:.0f}, BP {matched_bp:.0f}
- HCD totals this year: CO {h_co_total:.0f}, BP {h_bp_total:.0f}

## Unmatched HCD-row classification
- matched (any tier): {len(matched_df)}
- multi_row_same_apn: {n_multirow} (HCD splits a parcel D5 represents with one master)
- year_shifted (HCD year != D5 routed year): {sum(yearshift.values())} [{ys_str}]
- no_cpra_presence: {len(no_presence)}

## Method
Two tiers, injective. Tier 1: normalized tracking-ID equality (strip whitespace/case,
strip -REV/-DEF suffix). Tier 2: APN equality (current then prior) against still-unused
CPRA permits. CY2025 HCD rows deduped (draft+final doubling) before matching.
""")

    summary_rows.append({
        "year": year, "h_rows": len(h_year), "c_rows": len(c_year),
        "matched": len(matched_df), "unmatched_h": len(h_un), "unmatched_c": len(c_un),
        "multi_row": n_multirow, "year_shift": int(sum(yearshift.values())),
        "matched_co_units": float(matched_co), "matched_bp_units": float(matched_bp),
        "h_co_total": float(h_co_total), "h_bp_total": float(h_bp_total),
    })
    print(f"CY{year}: matched={len(matched_df)}, h_unmatched(no_presence)={len(h_un)}, "
          f"c_unmatched={len(c_un)}, multi_row={n_multirow}, year_shift={int(sum(yearshift.values()))}")
print("\nYear loop complete. 8 reconciliation directories written under data/audit/.")

/var/folders/zr/1lcy71z97n33bq1zyg3vtbp80000gn/T/ipykernel_9916/2490239590.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cf = pd.concat([fa, fb], ignore_index=True)


CPRA full-corpus distinct APNs: 13272
CY2018: matched=144, h_unmatched(no_presence)=6, c_unmatched=263, multi_row=9, year_shift=57
CY2019: matched=182, h_unmatched(no_presence)=6, c_unmatched=205, multi_row=15, year_shift=53
CY2020: matched=182, h_unmatched(no_presence)=11, c_unmatched=147, multi_row=13, year_shift=29


CY2021: matched=204, h_unmatched(no_presence)=12, c_unmatched=208, multi_row=13, year_shift=31
CY2022: matched=179, h_unmatched(no_presence)=13, c_unmatched=462, multi_row=7, year_shift=45
CY2023: matched=199, h_unmatched(no_presence)=14, c_unmatched=729, multi_row=13, year_shift=31
CY2024: matched=182, h_unmatched(no_presence)=8, c_unmatched=974, multi_row=20, year_shift=18
CY2025: matched=197, h_unmatched(no_presence)=11, c_unmatched=974, multi_row=10, year_shift=16

Year loop complete. 8 reconciliation directories written under data/audit/.


## Cross-year summary

In [5]:
# Cell 10 — cross-year summary table
summary = pd.DataFrame(summary_rows)[
    ["year","h_rows","c_rows","matched","unmatched_h","unmatched_c",
     "multi_row","year_shift","matched_co_units","matched_bp_units","h_co_total","h_bp_total"]]
D7_OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(D7_OUT / "cross_year_summary.csv", index=False)
print(summary.to_string(index=False))
print(f"\nSaved: {D7_OUT / 'cross_year_summary.csv'}")

 year  h_rows  c_rows  matched  unmatched_h  unmatched_c  multi_row  year_shift  matched_co_units  matched_bp_units  h_co_total  h_bp_total
 2018     216     407      144            6          263          9          57             222.0             276.0       229.0       380.0
 2019     256     387      182            6          205         15          53             299.0             308.0       313.0       363.0
 2020     235     329      182           11          147         13          29             396.0             721.0       405.0       766.0
 2021     260     412      204           12          208         13          31             314.0             464.0       331.0       506.0
 2022     244     641      179           13          462          7          45             782.0             603.0       828.0       887.0
 2023     257     928      199           14          729         13          31             689.0             421.0       716.0       432.0
 2024     228    115

## Regression test: CY 2024 vs committed ledger

In [6]:
# Cell 12 — regression: freshly-generated CY2024 vs the committed ledger snapshot.
# Label renames are expected and treated as cosmetic: old match_tier "1" == new
# "tier_1_tracking_id" (both tier level 1); old cpra_presence "no CPRA presence anywhere"
# == new "no_cpra_presence". Comparisons are on row-id sets, tier level, and numeric units.
print("="*64); print("REGRESSION TEST — CY 2024"); print("="*64)
checks = []
def check(name, ok, detail=""):
    checks.append(ok); print(f"  [{'PASS' if ok else 'FAIL'}] {name}{(' — ' + detail) if detail else ''}")

cy24 = AUDIT_BASE / "cy2024_reconciliation"
new_m = pd.read_csv(cy24 / "matched_pairs.csv")
new_h = pd.read_csv(cy24 / "h_unmatched.csv")
new_c = pd.read_csv(cy24 / "c_unmatched.csv")
old_m, old_h, old_c = OLD_2024["matched_pairs"], OLD_2024["h_unmatched"], OLD_2024["c_unmatched"]

def tier_level(v):
    s = str(v)
    if s in ("1", "tier_1_tracking_id"): return 1
    if s.startswith("tier_2"): return 2
    return 0

if old_m is None:
    check("committed matched_pairs available", False, "snapshot missing — cannot regress")
else:
    check("matched row count == 182", len(new_m) == 182, f"new={len(new_m)}, old={len(old_m)}")
    new_ids, old_ids = set(new_m["hcd_row_id"]), set(old_m["hcd_row_id"])
    check("matched hcd_row_id set identical", new_ids == old_ids,
          f"new-only={len(new_ids-old_ids)}, old-only={len(old_ids-new_ids)}")
    new_tl = {r.hcd_row_id: tier_level(r.match_tier) for r in new_m.itertuples()}
    old_tl = {r.hcd_row_id: tier_level(r.match_tier) for r in old_m.itertuples()}
    mismatch = [k for k in new_tl if k in old_tl and new_tl[k] != old_tl[k]]
    check("match tier-level identical (label-rename tolerant)", len(mismatch) == 0,
          f"{len(mismatch)} rows differ in tier level")

# 1951 Shattuck (hcd_row_id 1407) — tier 2, co 163/163
sh = new_m[new_m["hcd_row_id"] == 1407]
if len(sh):
    r = sh.iloc[0]
    ok = (tier_level(r["match_tier"]) == 2 and abs(float(r["co_units_hcd"])-163) < 0.5
          and abs(float(r["co_units_d5"])-163) < 0.5)
    check("1951 Shattuck (1407): tier2, co 163/163", ok,
          f"tier={r['match_tier']}, co_hcd={r['co_units_hcd']}, co_d5={r['co_units_d5']}")
else:
    check("1951 Shattuck (1407) present in matched", False, "row 1407 not found")

# h_unmatched: 8 rows, all no_presence, same id set
check("h_unmatched count == 8", len(new_h) == 8, f"new={len(new_h)}")
check("h_unmatched all no_cpra_presence", (new_h["cpra_presence"] == "no_cpra_presence").all()
      if len(new_h) else False)
if old_h is not None:
    check("h_unmatched hcd_row_id set identical to committed",
          set(new_h["hcd_row_id"]) == set(old_h["hcd_row_id"]),
          f"new={sorted(new_h['hcd_row_id'])}")

# c_unmatched: 974 rows
check("c_unmatched count == 974", len(new_c) == 974, f"new={len(new_c)}")
if old_c is not None:
    check("c_unmatched JURS_TRACKING_ID set identical to committed",
          set(new_c['JURS_TRACKING_ID'].astype(str)) == set(old_c['JURS_TRACKING_ID'].astype(str)))

print("="*64)
if all(checks):
    print(f"REGRESSION: ALL {len(checks)} CHECKS PASS — CY2024 faithfully reproduced.")
else:
    print(f"REGRESSION: {sum(checks)}/{len(checks)} pass, {len(checks)-sum(checks)} FAIL — "
          f"!!! INVESTIGATE before trusting other years / committing !!!")
print("="*64)

REGRESSION TEST — CY 2024
  [PASS] matched row count == 182 — new=182, old=182
  [PASS] matched hcd_row_id set identical — new-only=0, old-only=0
  [PASS] match tier-level identical (label-rename tolerant) — 0 rows differ in tier level
  [PASS] 1951 Shattuck (1407): tier2, co 163/163 — tier=tier_2_apn_current, co_hcd=163.0, co_d5=163.0
  [PASS] h_unmatched count == 8 — new=8
  [PASS] h_unmatched all no_cpra_presence
  [PASS] h_unmatched hcd_row_id set identical to committed — new=[1304, 1305, 1402, 1481, 2081, 2083, 2084, 2085]
  [PASS] c_unmatched count == 974 — new=974
  [PASS] c_unmatched JURS_TRACKING_ID set identical to committed
REGRESSION: ALL 9 CHECKS PASS — CY2024 faithfully reproduced.
